# Fastbox signal cube on the cropped L2021 grid

Generates a simulated H I brightness-temperature cube **on the same grid as the
cropped MeerKAT data**, and turns it into the signal-covariance starting point
`S_starting_point_cropped.npy` used by the Gibbs sampler.

The grid is set in **one place** — the `CROP` constant in section 1. Everything
downstream (`shape`, the survey geometry, `box_dims`, the *k*-binning and `S`)
is derived from it, so changing the crop needs no other edits. Keep `CROP`
identical to the one in `Sampling_Data_Gibbs.ipynb`, or `S` will be mismatched
to its own *k*-bins.

This supersedes `Gibbs_2026/fastbox_gen.ipynb`, which built a cubic 72^3 box
and used a `box_dims` that disagreed with its own `box_scale` by 1000x.

**Requires the non-cubic `fastbox` patch** (`nsamp` accepting a 3-tuple).
Upstream fastbox only supports cubic grids and will raise
`TypeError: 'tuple' object cannot be interpreted as an integer` here.

In [ ]:
# Run from the repo without installing: `pip install -e ..` makes this unnecessary.
import sys, pathlib
_root = pathlib.Path.cwd().parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

import json

import numpy as np
import numpy.fft as fft
import matplotlib.pyplot as plt

import fastbox
from fastbox.box import CosmoBox, default_cosmo

from imgibbs import (
    signal_covariance_sampler as SCS,
    bin_it, kbins_from_crop, survey_grid, data_path, load_l2021_cube,
    PIX_DEG,
)

np.random.seed(41)


## 1. The grid

`CROP` selects the sub-volume of `L2021_polished_cube.npy` (133, 73, 500) to
simulate on. The default is the tight bounding box of the drift-scan
footprint, `[33:103, 14:59, :]` → (70, 45, 500): it discards **no** valid
voxels while raising the fill fraction from 19.2% to 59.1%, so the sampler
inpaints flagged pixels rather than vast empty padding.

Tighter crops trade a little data for a lot of fill — see the "Future grid
options" table in `HANDOVER.md` before picking one.

In [ ]:
# ====================== SET THE GRID HERE ==============================
# This must match the CROP in 2_gibbs_sampling.ipynb. S_starting_point is
# indexed by k-bins derived from this grid, so a mismatch silently attaches
# the signal prior to the wrong wavenumbers.
CROP = (slice(33, 103), slice(14, 59), slice(0, 250))     # -> (70, 45, 250)
# =======================================================================

full_cube = load_l2021_cube()
data_cube = full_cube[CROP]

shape = data_cube.shape
valid = data_cube != 0
n_full, n_crop = int((full_cube != 0).sum()), int(valid.sum())

print(f'full cube     : {full_cube.shape}   fill {(full_cube != 0).mean()*100:.2f}%')
print(f'cropped       : {shape}   fill {valid.mean()*100:.2f}%')
print(f'valid voxels  : {n_full:,} -> {n_crop:,}  (lost {n_full - n_crop:,})')
print(f'grid voxels   : {full_cube.size:,} -> {data_cube.size:,}')


## 2. Survey geometry

Derived from the constants above: the *L* band spans 856-1712 MHz over 4096
channels, the polished cube holds channels 550-1050 (Wang et al.'s low
sub-band), and the map pixel size is 0.3 deg.

`box_dims` is computed from `shape`, so it tracks `CROP` automatically and
cannot drift out of step with the cube it describes.

The angular extent is that of the **FFT grid**, not the ~200 deg^2 scanned
footprint — the FFT runs over the whole array, and the footprint mask is
handled separately by the sampler's `w` weights.

In [ ]:
# All of the geometry comes from CROP via imgibbs.grid -- one derivation shared
# with the sampler and the PCA benchmark, so the three cannot disagree.
grid = survey_grid(CROP, shape)

box_dims = grid.box_dims
z_lo, z_hi, z_mid = grid.z_lo, grid.z_hi, grid.z_mid
freqs, chans = grid.freqs, grid.chans

print(grid.summary())


Note the voxels are strongly anisotropic — about 8 Mpc transverse against
1 Mpc radial — so the k-grid reaches roughly 8x further along the line of
sight than across it. This is precisely what a cubic `box_dims` got wrong.

A caveat on `Lx`/`Ly`: a single `D_M` at the mid redshift treats the volume
as a box, but the survey is really a frustum — the same 21 deg subtends
about 465 Mpc at *z* = 0.32 and 647 Mpc at *z* = 0.46. Fine for a prior and
for a P(k) comparison; revisit if you need precision P(k).

## 3. Simulated H I signal cube

The full chain from `fastbox_gen.ipynb`: Gaussian density -> H I bias ->
log-normal transform -> redshift-space distortions -> brightness temperature.

`HITracer.signal_amplitude()` returns **mK**, whereas the MeerKAT cube is in
**K**, so we convert. Getting this wrong would misscale `S` by 10^6.

In [ ]:
box = CosmoBox(cosmo=default_cosmo, box_scale=box_dims, nsamp=shape,
               redshift=z_mid, realise_now=False)
print(f'box.N     = {box.N}      (tuple -> non-cubic grid)')
print(f'box.shape = {box.shape}')

# (a) Gaussian density field
box.realise_density()

# (b) scale by the H I bias
tracer = fastbox.tracers.HITracer(box)
delta_hi = box.delta_x * tracer.bias_HI()

# (c) log-normal transform
delta_ln = box.lognormal(delta_hi)

# (d) radial velocity field (from the Gaussian density field)
vel_k = box.realise_velocity(delta_x=box.delta_x, inplace=True)
vel_z = fft.ifftn(vel_k[2]).real

# (e) into redshift space
delta_s = box.redshift_space_density(delta_x=delta_ln.real, velocity_z=vel_z,
                                     sigma_nl=120., method='linear')

# (f) brightness temperature, mK -> K to match the data cube
Tb_mK = tracer.signal_amplitude()
signal_cube = (Tb_mK * (1. + delta_s)) / 1000.0

print(f'\nTb(z={z_mid:.3f})   = {Tb_mK:.4f} mK')
print(f'signal_cube     : {signal_cube.shape}')
print(f'  mean {signal_cube.mean():.3e} K   std {signal_cube.std():.3e} K')
print(f'  data cube mean {data_cube[data_cube!=0].mean():.3f} K '
      f'(foreground dominated, ~10^4 x larger)')

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4), dpi=120)

im = ax[0].imshow(signal_cube[:, :, 249].T, origin='lower', cmap='inferno',
                  aspect='auto')
plt.colorbar(im, ax=ax[0], label='T$_b$ [K]')
ax[0].set_title('Simulated HI signal, channel 250')
ax[0].set_xlabel('RA pixel'); ax[0].set_ylabel('Dec pixel')

ax[1].plot(freqs, signal_cube[35, 22, :], lw=0.8)
ax[1].set_xlabel('Frequency (MHz)'); ax[1].set_ylabel('T$_b$ [K]')
ax[1].set_title('Line of sight through pixel (35, 22)')

plt.tight_layout(); plt.show()

## 4. k-binning

`make_kbins` builds the radial |k| bins for a general (Nx, Ny, Nz) grid, and
`bin_it` groups Fourier modes by bin. Both are shared with the sampling
notebook and **must be given the same `box_dims`** — the voxel-to-bin mapping
is only invariant under an overall rescaling for a *cubic* box, so with an
anisotropic box the axis ratios matter.

The covariance sampler asserts more than 2 modes per bin, so the bin count is
limited by the lowest-*k* bin. On the default (70, 45, 500) grid 14 bins leaves
only 2 modes in the first bin and trips the assert; **12 bins** is the most that
works, with 8 modes in the lowest bin.

That limit depends on the grid, so if you change `CROP`, set `n_k_bins = None`
below to have the largest workable value picked for you.

In [ ]:
# was make_kbins + a hand-tuned n_k_bins, with k_min taken from the
# BOX. The cube is a bounding box round a diagonal band filling ~59% of
# it, so modes longer than the band are set by the zero padding rather
# than by data, and on the 250-channel cut the lowest bin held 4 modes,
# all with kz=0 (frequency-constant, degenerate with the foreground).
# kbins_from_crop takes k_min from the measured footprint, drops the
# kz=0 plane and sets the bin count from occupancy.
#
# MUST match Sampling_Data_Gibbs.ipynb -- S_starting_point_cropped.npy
# is indexed by these bins. Same call, same data_cube, same box_dims.
n_k_bins = 5#12          # now a CEILING; the function may return fewer

sig_k, idxs, kbin_meta = kbins_from_crop(data_cube, box_dims, pix_deg=PIX_DEG,
                                         max_bins=n_k_bins)
n_k_bins = kbin_meta['n_k_bins']  # the count actually chosen
counts   = np.array(kbin_meta['modes_per_bin'])

print(f'n_k_bins = {n_k_bins}')
print(f'modes per bin : {counts}')
print(f'lowest bin    : {counts.min()} modes '
      f'({"OK" if counts.min() >= kbin_meta["min_modes"] else "TOO FEW"})')


## 5. Signal covariance starting point

`SCS` draws P(k) per bin from the inverse-gamma conditional posterior. We
broadcast that back onto every voxel via `idxs` to build the diagonal `S`,
leaving the DC mode with a large variance so it stays unconstrained.

In [ ]:
# Mean-subtract, then FFT once (orthonormal, matching the sampler's convention)
s_fft = np.fft.fftn(signal_cube - signal_cube.mean(), norm='ortho')

binned_s, k_bins = bin_it(s_fft, sig_k, idxs)
_, PkSample = SCS(np.concatenate(binned_s), np.concatenate(k_bins))
PkSample = np.asarray(PkSample)

# Directly measured P(k) too - the truth curve to compare sampler output against
Pk_true = np.array([np.mean(np.abs(b)**2) for b in binned_s])

# Broadcast the sampled P(k) onto the voxel grid
S = np.zeros(len(idxs))
for gg, bin_idx in enumerate(np.unique(idxs)[1:]):
    S[idxs == bin_idx] = PkSample[gg]
# idxs==0 now covers DC + the kz=0 plane + sub-footprint modes, not
# just DC. A flat 1e30 prior across that plane would let the signal
# absorb foreground power in exactly the degenerate modes, so suppress
# them and leave only the true DC mode free to carry the mean.
S[idxs == 0] = 1e-12 * np.median(PkSample)
S[kbin_meta['dc_index']] = 1e30

print(f'S            : {S.shape}  (expect {np.prod(shape):,})')
print(f'unique values: {len(np.unique(S))}  (expect {n_k_bins} bins + DC)')
print(f'S range (excl. DC): {S[idxs>0].min():.3e} - {S[idxs>0].max():.3e} K^2')

In [ ]:
np.save(data_path('S_starting_point_cropped.npy'), S)
np.save(data_path('Fastbox_Pk_cropped.npy'), Pk_true)
np.save(data_path('Fastbox_PkSample_cropped.npy'), PkSample)
np.save(data_path('Fastbox_kbins_cropped.npy'), sig_k)
np.save(data_path('Fastbox_cube_cropped.npy'), signal_cube)

# Metadata is built from CROP rather than copied from an input file, so it
# always describes the grid this notebook actually ran on.
bounds = {ax: list(CROP[i].indices(full_cube.shape[i])[:2])
          for i, ax in enumerate('xyz')}

meta_out = {'source': 'L2021_polished_cube.npy',
            'fits_source': 'Nscan961_Tsky_cube_p0.3d_sigma4.0_iter2.fits',
            'fits_channels': [int(chans[0]), int(chans[-1]) + 1],
            'crop': bounds,
            'shape': list(shape),
            'pixel_deg': PIX_DEG,
            'box_dims': [float(v) for v in box_dims],
            'z_range': [float(z_lo), float(z_hi)],
            'z_mid': float(z_mid),
            'freq_MHz': [float(freqs[0]), float(freqs[-1])],
            'n_k_bins': n_k_bins,
            # record how the bins were chosen so the Gibbs notebook can
            # assert it is using the same ones S was built with.
            'kbin_k_min': kbin_meta['k_min'],
            'kbin_k_max': kbin_meta['k_max'],
            'kbin_drop_kz0': kbin_meta['drop_kz0'],
            'kbin_kmin_axis': kbin_meta['kmin_axis'],
            'kbin_modes_per_bin': kbin_meta['modes_per_bin'],
            'kbin_sig_k': kbin_meta['sig_k'],
            'redshift': float(z_mid),
            'Tb_mK': float(Tb_mK),
            'units': 'K (signal cube and S in K^2)'}
with open(data_path('S_starting_point_cropped_meta.json'), 'w') as f:
    json.dump(meta_out, f, indent=2)

print('saved:')
for n in [data_path('S_starting_point_cropped.npy'), data_path('Fastbox_Pk_cropped.npy'),
          data_path('Fastbox_PkSample_cropped.npy'), data_path('Fastbox_kbins_cropped.npy'),
          data_path('Fastbox_cube_cropped.npy'), data_path('S_starting_point_cropped_meta.json')]:
    print('  ' + n)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5), dpi=120)
ax.plot(sig_k, Pk_true, 'o-', color='crimson', lw=1.5, ms=5,
        label='Measured P(k) from the fastbox cube')
ax.plot(sig_k, PkSample, 's--', color='mediumvioletred', lw=1.2, ms=4,
        label='Inverse-gamma sample (used for S)')
ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlabel('k [Mpc$^{-1}$]'); ax.set_ylabel('P(k) [K$^2$]')
ax.set_title(f'Simulated HI P(k), {shape} grid at z={z_mid:.3f}')
ax.legend(); plt.tight_layout(); plt.show()

## Next steps

In `Sampling_Data_Gibbs.ipynb` (and `sampling_gibbs.py`, which is still on the
uncropped cube), set `CROP` to the value printed by section 2:

```python
CROP = (slice(33, 103), slice(14, 59), slice(None))     # -> (70, 45, 500)
```

Both notebooks derive `box_dims` from `CROP` the same way, so matching the crop
is enough — there is no longer a separate `box_dims` literal to keep in step.
`n_k_bins` **must** still match, or `S` will be mismatched to its own *k*-bins.

The `LP_fg_model_cube.npy` foreground model is still on the old 72^3 slice; the
sampling notebook rebuilds its Legendre model inline instead.